<a href="https://colab.research.google.com/github/tahumada/MSO-AEON/blob/main/DECam/DECam_submit_individual_target2AEON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DECam target submission to the AEON queue
Tomas Ahumada - tomas.ahumada@noirlab.edu

Last modified: Aug 2026

This code intends to be a tutrorial for NOIRLab-AEON users of the Dark Energy Camera (DECam) mounted at the 4m V. M. Blanco telescope. The AEON queue is run by the Las Cumbres Observatory (LCO) Scheduler, thus the user requires an active account to access the LCO portal and generate a LCO key to submit requests to an active program.

Information about DECam can be found here: https://noirlab.edu/science/programs/ctio/instruments/Dark-Energy-Camera
Information about LCO can be found here: https://observe.lco.global/

Once you have an active user in the LCO portal, you can find the API key here https://observe.lco.global/accounts/profile

# Outline
1. Define target parameters
2. Get template request (json format) - this json file is modified and later sent to the LCO queue
3. Make the payload, modifying the json template.
4. Submit target individually. The target can have multiple filters.

In [30]:
import copy
from datetime import datetime, timedelta
import glob
import io
import json
import os
import re
import time
import urllib.request
from urllib.parse import urlparse
from astropy.time import Time
import numpy as np
import pandas as pd
import requests

In [35]:
# get example json
def get_example(example_path):
    if "github.com" in example_path and "/blob/" in example_path:
        example_path = example_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    if urlparse(example_path).scheme in ('http', 'https'):
        response = requests.get(example_path)

        if response.status_code == 200:
            return response.json()
        else:
            raise Exception(f"Failed to fetch file from URL. Status code: {response.status_code}")
    else:
        pass



In [41]:
template_json_url = "https://raw.githubusercontent.com/tahumada/MSO-AEON/main/DECam/example.json"
print('The template of an observation:')
get_example(template_json_url)

The template of an observation:


{'id': 2495241,
 'requests': [{'id': 4168585,
   'location': {'telescope_class': '4m0'},
   'configurations': [{'id': 14811742,
     'constraints': {'max_airmass': 1.6,
      'min_lunar_distance': 30.0,
      'max_lunar_phase': 1.0,
      'max_seeing': None,
      'min_transparency': None,
      'extra_params': {}},
     'instrument_configs': [{'exposure_time': 40.0,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 0, 'offset_dec': 0}}],
     'acquisition_config': {'mode': 'OFF',
      'exposure_time': None,
      'extra_params': {}},
     'guiding_config': {'optional': True,
      'mode': 'ON',
      'optical_elements': {},
      'exposure_time': None,
      'extra_params': {}},
     'target': {'type': 'ICRS',
      'name': 'test2',
      'ra': 151.197,
      'dec': 7.801,
      'proper_motion_ra': 0.0,
      'proper_motion_dec': 0.0,
      'parallax': 0.0,
      'epoch': 2

In [42]:
# Defining the variables you need to change:

PROPOSAL_ID  = 'HERE YOUR PROPID'
OBSERVATION_TYPE = 'NORMAL' # can be 'TIME_CRITICAL' or 'RAPID_RESPONSE'
WINDOW_START = '2026-08-01 17:32:00'
WINDOW_END   = '2026-09-01 17:32:00'
NAME = 'test'
RA = '15'
DEC = '-20'
FILTER_LIST = ['g','r','i','z']
EXPTIME_LIST = [30,60,90,120]
EXPOSURE_COUNT_LIST = [1,2,3,1]
MAX_AIRMASS = 1.8
MIN_MOON = 30
DETECTOR_CENTERING = 'S4'
DITHER = 'None' # currently dithers can be assigned through the LCO portal

variables = {
        "proposal": PROPOSAL_ID,
        'observation_type': OBSERVATION_TYPE,
        'maximum_airmass': MAX_AIRMASS,
        'minimum_lunar_distance': MIN_MOON,
        'detector_centering':  DETECTOR_CENTERING,
        'dither': DITHER,
        'windows': [{'start': WINDOW_START, 'end': WINDOW_END}],
        'target': {'id': NAME,
                   'ra': RA,
                   'dec': DEC,
                   'filters': FILTER_LIST,
                   'exptime': EXPTIME_LIST,
                   'exposure_count': EXPOSURE_COUNT_LIST
                   }
    }



data = get_example(template_json_url)

req_config = data['requests'][0]['configurations'][0]

req_config['constraints']['max_airmass'] = variables['maximum_airmass']
req_config['constraints']['minimum_lunar_distance'] = variables['minimum_lunar_distance']

# Safely formatting the date (zero-padding months and days)
# now = Time.now().datetime
# date = f"{now.year}{now.month:02d}{now.day:02d}"
# data['name'] = f"{variables['target']['id']}_{date}"

data['name'] = variables['target']['id'] # name of your request
data['proposal'] = variables['proposal']
data['observation_type'] = variables['observation_type']
data['requests'][0]['windows'] = variables['windows']

req_config['target']['name'] = variables['target']['id'] # name of your target
req_config['target']['ra'] = str(variables['target']['ra'])
req_config['target']['dec'] = str(variables['target']['dec'])
req_config['extra_params']['detector_centering'] = variables['detector_centering']

# base instrument configuration template
base_instrument_config = req_config['instrument_configs'][0]

# Reset filter configuration in the payload
req_config['instrument_configs'] = []

# loop through filters, copy, and append
for filt in ['g', 'r', 'i', 'z']:
    if filt in variables['target']['filters']:
        new_config = copy.deepcopy(base_instrument_config)

        new_config['optical_elements']['filter'] = filt
        idx = list(variables['target']['filters']).index(filt)

        new_config['exposure_time'] = float(variables['target']['exptime'][idx])
        new_config['exposure_count'] = float(variables['target']['exposure_count'][idx])
        req_config['instrument_configs'].append(new_config)


data

{'id': 2495241,
 'requests': [{'id': 4168585,
   'location': {'telescope_class': '4m0'},
   'configurations': [{'id': 14811742,
     'constraints': {'max_airmass': 1.8,
      'min_lunar_distance': 30.0,
      'max_lunar_phase': 1.0,
      'max_seeing': None,
      'min_transparency': None,
      'extra_params': {},
      'minimum_lunar_distance': 30},
     'instrument_configs': [{'exposure_time': 30.0,
       'optical_elements': {'filter': 'g'},
       'mode': 'default',
       'exposure_count': 1.0,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 0, 'offset_dec': 0}},
      {'exposure_time': 60.0,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 2.0,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 0, 'offset_dec': 0}},
      {'exposure_time': 90.0,
       'optical_elements': {'filter': 'i'},
       'mode': 'default',
       'exposure_count': 3.0,
       'rotator_mode': '',
       'extra_params': {'offset_ra'

In [43]:
LCO_TOKEN ='HERE YOUR LCO TOKEN'
requestpath  = "https://observe.lco.global/api/requestgroups/"

# saving the status of the request
lco_sent,lco_failed = [],[]

# change this if you do not want to submit the request, and just see how the payload looks
send = True

# send as test
if send:
  response = requests.post(
          requestpath,
          headers={"Authorization": f"Token {LCO_TOKEN}"},
          json=data,  # Make sure you use json!
      )

  if response.status_code == 400:
      print(variables['target']['id'], 'Failed (sending to queue)')
      lco_failed.append([variables['target']['id'],response.text])

  elif response.status_code == 201 or response.status_code == 200:
      lco_sent.append([variables['target']['id'],'sent!',response.json()['id']])



In [44]:
print('sources sent:', lco_sent)


sources sent: [['test', 'sent!', 2644968]]


In [45]:
print('sources failed:',lco_failed)

sources failed: []


In [46]:
ids = np.asarray(lco_sent).T[-1]
names = np.asarray(lco_sent).T[0]

for i,idlco in enumerate(ids):
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    if response.status_code == 200:
      print('request for:', names[i], 'cancelled\nid:', idlco)
    else:
      print('Cancel request failed for ', names[i])

request for: test cancelled
id: 2644968
